<a href="https://colab.research.google.com/github/cthomps9/5300/blob/main/Temporal_Residual_Transformer_(TRT)_Model_Implementation_Golden_child.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, precision_recall_curve, auc, roc_curve, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score, ConfusionMatrixDisplay
import time

from google.colab import drive
drive.mount('/content/drive')

csv_path = "/content/drive/MyDrive/Erdos Deep Learning/subset_4000_patients/subset_train.csv"
df = pd.read_csv(csv_path)

# Step 1: Data Processing Utilities

def preprocess_data_into_segments(df, label_col_index, extra_exclusions):
    """
    Splits the DataFrame into a list of variable-length segments based on
    changes in the Subject ID (fixed at index 0).
    The feature columns for the segment are dynamically determined by excluding:
    1. The Subject ID column (index 0).
    2. The target label column (index specified by label_col_index).
    3. All columns listed in extra_exclusions.
    """

    label_column_name = df.columns[label_col_index]

    # Segmentation is based on changes in the Subject ID, which is fixed at index 0
    segmentation_col_name = df.columns[0]

    # Identify indices where the segment changes (where the Subject ID changes)
    change_indices = df[df[segmentation_col_name] != df[segmentation_col_name].shift(1)].index.tolist()

    # Ensure starting index is included
    if change_indices and change_indices[0] != df.index[0]:
        change_indices = [df.index[0]] + change_indices
    elif not change_indices:
         change_indices = [df.index[0]]

    segments = []

    # --- Feature Exclusion Logic ---
    excluded_names = {segmentation_col_name, label_column_name}
    excluded_names.update(extra_exclusions)

    # Define the final feature columns by excluding the master list
    feature_columns = [col for col in df.columns if col not in excluded_names]
    # --- END Feature Exclusion Logic ---

    for i in range(len(change_indices)):
        start_index = change_indices[i]
        end_index = change_indices[i+1] if i + 1 < len(change_indices) else len(df)

        # Extract data
        segment_data = df.loc[df.index[start_index:end_index], feature_columns].values

        if len(segment_data) == 0:
            continue

        # --- CRITICAL: Patient-Level Sepsis Labeling ---
        # The segment label is the MAX value (i.e. 1 if Sepsis ever occurred, 0 otherwise)
        segment_labels_col = df.loc[df.index[start_index:end_index], label_column_name]
        segment_label = segment_labels_col.max()
        # --- END CRITICAL CHANGE ---

        segments.append({
            'data': segment_data.astype(np.float32),
            'label': segment_label,
            'length': len(segment_data)
        })

    return segments, feature_columns


def load_and_preprocess_data(file_path, label_col_name, extra_exclusions, batch_size=32, shuffle=True, return_subject_ids=False):
    """Loads, cleans, segments, and prepares a DataLoader for a single dataset."""

    try:
        df = pd.read_csv(file_path)

        if df.empty:
            raise ValueError("DataFrame is empty after loading. Check the input file.")

        # --- Data Cleaning ---
        subject_id_col_name = df.columns[0]

        # Check if the SepsisLabel column exists
        if label_col_name not in df.columns:
             raise ValueError(f"Missing required column: '{label_col_name}'. Please ensure your CSV file contains a column with this exact name.")

        # 1. Get the index of the SepsisLabel column
        LABEL_COLUMN_INDEX = df.columns.get_loc(label_col_name)

        # 2. Identify all sensor columns except Subject ID and SepsisLabel
        all_sensor_columns = [col for col in df.columns if col not in [subject_id_col_name, label_col_name]]

        # Clean numerical features
        for col in all_sensor_columns:
            # Coerce non-numeric values to NaN
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # Impute NaNs with 0.0 to preserve time sequence integrity
        df[all_sensor_columns] = df[all_sensor_columns].fillna(0.0)

        # 3. Clean the SepsisLabel column and ensure it is an integer (0 or 1)
        df[label_col_name] = df[label_col_name].astype(int)

        # 4. Define the final label map (for consistency, though fixed at 0/1)
        label_map = {0: 0, 1: 1}

        # --- Segmentation ---
        segments, actual_feature_columns = preprocess_data_into_segments(
            df,
            label_col_index=LABEL_COLUMN_INDEX,
            extra_exclusions=extra_exclusions
        )

        if not segments:
            raise ValueError("No valid segments were created after preprocessing.")

        print(f"File: {file_path} loaded. Total segments: {len(segments)}")

        # --- DataLoader Setup ---
        dataset = SegmentDataset(segments, label_map)
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            collate_fn=collate_variable_length
        )

        # Map for easy decoding (0 -> 'No Sepsis' or 1 -> 'Sepsis')
        id_to_label = {0: 'No Sepsis', 1: 'Sepsis'}

        if return_subject_ids:
            unique_ids = set(df[subject_id_col_name].unique())
            return loader, dataset, actual_feature_columns, id_to_label, unique_ids

        return loader, dataset, actual_feature_columns, id_to_label

    except Exception as e:
        print(f"FATAL ERROR loading/processing {file_path}: {e}")
        # Return Nones for all expected values
        if return_subject_ids:
             return None, None, None, None, None
        return None, None, None, None


class SegmentDataset(Dataset):
    """PyTorch Dataset for variable-length time-series segments."""
    def __init__(self, segments_list, label_map=None):
        self.segments = segments_list
        self.label_map = label_map

    def __len__(self):
        return len(self.segments)

    def __getitem__(self, idx):
        segment = self.segments[idx]

        # Data is already a numpy array from preprocessing
        data_tensor = torch.from_numpy(segment['data'])

        # Label is already a numerical ID (0 or 1)
        label_id = segment['label']
        label_tensor = torch.tensor(label_id, dtype=torch.long)

        # The collate function handles returning the lengths
        return data_tensor, label_tensor

def collate_variable_length(batch):
    """
    Collate function for DataLoader: pads variable-length sequences to the max
    length in the batch and returns original lengths.
    """
    data = [item[0] for item in batch]  # List of tensors
    labels = [item[1] for item in batch] # List of scalar label tensors

    # Pad data sequences to the length of the longest sequence in the batch
    # padded_data shape: (Batch, Max_Length, Features)
    padded_data = pad_sequence(data, batch_first=True, padding_value=0.0)

    # Stack the labels
    stacked_labels = torch.stack(labels)

    # Calculate and return original lengths
    lengths = torch.tensor([len(d) for d in data])

    return padded_data, stacked_labels, lengths

def calculate_metrics(y_true, y_pred, y_prob):
    """Calculates all binary classification metrics."""

    # --- Basic Metrics ---
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    # --- Discrimination Metrics ---
    try:
        # AUC-ROC: General measure of separability
        roc_auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc_auc = 0.0 # Undefined if only one class is present

    try:
        # Average Precision (AUPRC): Best for highly imbalanced data
        avg_prec = average_precision_score(y_true, y_prob)
    except ValueError:
        avg_prec = 0.0

    return {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'AUC-ROC': roc_auc,
        'Avg Precision': avg_prec
    }


def evaluate_model(model, loader, device, criterion=None):
    """Evaluates the model on a given dataset (validation or test) and returns metrics."""
    model.eval()
    total_loss = 0

    all_labels = []
    all_preds = []
    all_probs = [] # Raw probabilities for the positive class (Sepsis=1)

    with torch.no_grad():
        for data, labels, lengths in loader:
            data, labels, lengths = data.to(device), labels.to(device), lengths.to(device)

            outputs = model(data, lengths)

            if criterion:
                loss = criterion(outputs, labels)
                total_loss += loss.item()

            # Get predicted class (0 or 1)
            _, predicted = torch.max(outputs.data, 1)

            # Get probabilities for ROC/AUPRC this process uses softmax to convert logits)
            probs = F.softmax(outputs, dim=1)[:, 1] # Probability of class 1 (i.e Sepsis)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader) if criterion else None

    metrics = calculate_metrics(
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs)
    )

    return avg_loss, metrics

Mounted at /content/drive


In [ ]:
# Step 2: Temporal Residual Transformer Model

class TemporalTransformerBlock(nn.Module):
    """Single Residual Block combining Multi-Head Attention and a Feed-Forward Network."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        super().__init__()

        # Self-Attention Layer
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        # Feed Forward Layer
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        # 1. Self-Attention Sublayer with Residual connection
        attn_output, _ = self.self_attn(
            src, src, src,
            attn_mask=src_mask,
            key_padding_mask=src_key_padding_mask
        )
        # Add & Norm (Residual connection applied before normalization)
        src = src + self.dropout1(attn_output)
        src = self.norm1(src)

        # 2. Feed Forward Sublayer with Residual connection
        ff_output = self.linear2(F.relu(self.linear1(src)))

        # Add & Norm
        src = src + self.dropout2(ff_output)
        src = self.norm2(src)

        return src


class TemporalResidualTransformer(nn.Module):
    """
    Temporal Residual Transformer for sequence classification, handling variable
    lengths via padding masks.
    """
    def __init__(self, input_features, num_classes, d_model, nhead, num_layers, dim_feedforward, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # 1. Input Projection: Map raw features (F) to model dimensions (d_model)
        self.input_projection = nn.Linear(input_features, d_model)

        # 2. Stack Transformer Encoder Blocks
        encoder_layer = TemporalTransformerBlock(d_model, nhead, dim_feedforward, dropout)
        self.transformer_encoder = nn.ModuleList([encoder_layer for _ in range(num_layers)])

        # 3. Output Classifier
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, lengths):
        # x is (Batch, Max_Seq_Len, Features)

        # 1. Input Projection
        x = self.input_projection(x) # x is now (Batch, Max_Seq_Len, d_model)

        # 2. Create Mask for Padding (Crucial for variable length)
        max_len = x.size(1)

        # Creates a mask: True where data is padding, False where it is real data
        mask = torch.arange(max_len, device=x.device).unsqueeze(0) >= lengths.unsqueeze(1)

        # 3. Pass through all Transformer Blocks, ignoring padding
        for layer in self.transformer_encoder:
            # src_key_padding_mask is (Batch, Max_Seq_Len)
            x = layer(x, src_key_padding_mask=mask)

        # 4. Global Feature Aggregation (Masked Average Pooling)
        # We need to average only the non-padded tokens.

        # Invert mask: True where real data is, False where padding is
        real_mask = (~mask).float().unsqueeze(-1) # (Batch, Max_Seq_Len, 1)

        # Apply mask to data (padding becomes 0.0)
        masked_x = x * real_mask

        # Sum the non-padded tokens
        sum_features = masked_x.sum(dim=1) # (Batch, d_model)

        # Count the number of non-padded tokens per sample
        real_lengths_float = lengths.float().unsqueeze(-1) # (Batch, 1)

        # Calculate the average, ensuring no division by zero
        avg_features = sum_features / torch.clamp(real_lengths_float, min=1.0) # (Batch, d_model)

        # 5. Classification
        output = self.classifier(avg_features)

        return output

In [ ]:
# Step 3: Training Function and Execution

def train_trt_model():
    print("--- Starting Temporal Residual Transformer Training (Binary Sepsis Classification) ---")

    # --- NOTICE THIS IS THE FILE PATH PLEASE CHANGE BEFORE RUNNING ---
    BASE_PATH = '/content/drive/MyDrive/Erdos Deep Learning/subset_15000_patients'
    TRAIN_FILE_PATH = f'{BASE_PATH}/subset_train.csv'
    VAL_FILE_PATH = f'{BASE_PATH}/subset_val.csv'
    TEST_FILE_PATH = f'{BASE_PATH}/subset_test.csv'

    # --- CONSTANTS ---
    SEPSIS_LABEL_COL_NAME = "SepsisLabel"
    BATCH_SIZE = 32
    NUM_EPOCHS = 20
    NUM_CLASSES = 2

    # --- DEFINITION OF COLUMNS TO EXCLUDE FROM FEATURE INPUT ---
    COLUMNS_TO_EXCLUDE_EXTRA = [
        "EtCO2", "BaseExcess", "HCO3", "FIO2", "PaCO2",
        "Alkanline Phosphate", "Miladirect", "Bilirubin direct",
        "Bilirubin total", "Lactate", "Troponinl one", "PTT", "Fibrinogen"
    ]
    # ------------------END--------------------------------

    # --- Data Loading and Preprocessing ---

    print("\n--- Loading Training Data ---")
    train_loader, train_dataset, actual_feature_columns, id_to_label, train_ids = load_and_preprocess_data(
        TRAIN_FILE_PATH, SEPSIS_LABEL_COL_NAME, COLUMNS_TO_EXCLUDE_EXTRA, BATCH_SIZE, shuffle=True, return_subject_ids=True
    )

    print("\n--- Loading Validation Data ---")
    val_loader, val_dataset, _, _, val_ids = load_and_preprocess_data(
        VAL_FILE_PATH, SEPSIS_LABEL_COL_NAME, COLUMNS_TO_EXCLUDE_EXTRA, BATCH_SIZE, shuffle=False, return_subject_ids=True
    )

    print("\n--- Loading Test Data ---")
    test_loader, test_dataset, _, _, test_ids = load_and_preprocess_data(
        TEST_FILE_PATH, SEPSIS_LABEL_COL_NAME, COLUMNS_TO_EXCLUDE_EXTRA, BATCH_SIZE, shuffle=False, return_subject_ids=True
    )

    # Check to see if all loaders were successfully created
    if not all([train_loader, val_loader, test_loader]):
        print("Training cannot start due to critical data loading errors.")
        return

    # --- Subject ID Overlap Check ---
    val_overlap = train_ids.intersection(val_ids)
    test_overlap = train_ids.intersection(test_ids)

    if val_overlap or test_overlap:
        print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
        print("!! WARNING: DATA LEAKAGE DETECTED (Patient-Specific Overfitting Risk) !!")
        if val_overlap:
            print(f"!! {len(val_overlap)} Subject IDs present in BOTH Training and Validation sets.")
        if test_overlap:
            print(f"!! {len(test_overlap)} Subject IDs present in BOTH Training and Test sets.")
        print("!! High validation/test accuracy may be due to the model memorizing patients.")
        print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")
    else:
        print("\n--- Subject ID Integrity Check: PASS ---")


    # --- Model and Training Setup ---
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    INPUT_FEATURES = len(actual_feature_columns)

    print(f"\nModel Input Features (Sensor Columns Used): {INPUT_FEATURES}")
    print(f"Classes: {id_to_label}")

    # --- Calculate Class Weights for Imbalanced Data ---
    train_labels = [segment['label'] for segment in train_loader.dataset.segments]

    # Count occurrences of each class (0: No Sepsis, 1: Sepsis)
    class_counts = pd.Series(train_labels).value_counts().sort_index()

    # Check for missing classes (e.g., if train set has no Sepsis cases)
    if len(class_counts) < NUM_CLASSES:
        print("WARNING: Training set is missing one or more classes. Cannot calculate weights.")
        return

    # Calculate weights: Inverse of class frequency (Total Samples / Class Count)
    total_samples = len(train_labels)
    weights = total_samples / class_counts.values

    # Normalize weights so they sum up to the number of classes
    weights = weights / np.sum(weights) * NUM_CLASSES

    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"Calculated Class Weights (0=No Sepsis, 1=Sepsis): {class_weights.tolist()}")
    # --- END Weight Calculation ---


    model = TemporalResidualTransformer(
        input_features=INPUT_FEATURES,
        num_classes=NUM_CLASSES,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=256
    ).to(DEVICE)

    # Apply weighted loss function
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # --- Training Loop ---
    print(f"\nStarting training on device: {DEVICE}")
    start_time = time.time()

    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0

        # Training Step
        for batch_idx, (data, labels, lengths) in enumerate(train_loader):
            data, labels, lengths = data.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(data, lengths)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_avg_loss = total_loss / len(train_loader)

        # --- Validation Step ---
        val_avg_loss, val_metrics = evaluate_model(model, val_loader, DEVICE, criterion)

        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] | Train Loss: {train_avg_loss:.4f} | Val Loss: {val_avg_loss:.4f}, Acc: {val_metrics["Accuracy"]:.4f}, Prec: {val_metrics["Precision"]:.4f}, Rec: {val_metrics["Recall"]:.4f}, AUC: {val_metrics["AUC-ROC"]:.4f}')


    end_time = time.time()
    print(f"\nTraining complete in {end_time - start_time:.2f} seconds.")

    # --- Final Test Set Evaluation ---
    print("\n--- Test Set Evaluation ---")
    test_avg_loss, test_metrics = evaluate_model(model, test_loader, DEVICE, criterion)

    print(f"Test Loss: {test_avg_loss:.4f}")
    print(f"Test Accuracy: {test_metrics['Accuracy']:.4f}")
    print(f"Test Precision: {test_metrics['Precision']:.4f}")
    print(f"Test Recall (Sensitivity): {test_metrics['Recall']:.4f}")
    print(f"Test F1 Score: {test_metrics['F1 Score']:.4f}")
    print(f"Test AUC-ROC: {test_metrics['AUC-ROC']:.4f}")
    print(f"Test Avg Precision (AUPRC): {test_metrics['Avg Precision']:.4f}")


    # --- Single Sample Inference Example from Test Set ---
    model.eval()

    try:
        # Get the first sample from the test dataset
        test_data_raw, test_label_id_raw = test_dataset[0]

        # Prepare the single sample for the model (add batch dimension and length)
        test_data = test_data_raw.unsqueeze(0).to(DEVICE) # (1, Seq_Len, Features)
        test_lengths = torch.tensor([test_data.size(1)]).to(DEVICE) # (1)

        with torch.no_grad():
            output = model(test_data, test_lengths)
            _, predicted_id = torch.max(output.data, 1)

        actual_label = id_to_label[test_label_id_raw.item()]
        predicted_label = id_to_label[predicted_id.item()]

        print(f"\nExample Inference (1st Test Sample):")
        print(f"Test Segment Length: {test_data.size(1)} rows.")
        print(f"Actual Label: {actual_label}")
        print(f"Predicted Label: {predicted_label}")
    except IndexError:
        print("\nSkipping inference test: Test dataset contains no segments.")


# Next line checks to see if the code is running in an interactive environment
try:
    # Checks for pandas
    import pandas as pd
    train_trt_model()
except Exception as e:
    print(f"An unexpected error occurred during final execution: {e}")

--- Starting Temporal Residual Transformer Training (Binary Sepsis Classification) ---

--- Loading Training Data ---
File: /content/drive/MyDrive/Erdos Deep Learning/subset_15000_patients/subset_train.csv loaded. Total segments: 7500

--- Loading Validation Data ---
File: /content/drive/MyDrive/Erdos Deep Learning/subset_15000_patients/subset_val.csv loaded. Total segments: 3000

--- Loading Test Data ---
File: /content/drive/MyDrive/Erdos Deep Learning/subset_15000_patients/subset_test.csv loaded. Total segments: 4500

--- Subject ID Integrity Check: PASS ---

Model Input Features (Sensor Columns Used): 54
Classes: {0: 'No Sepsis', 1: 'Sepsis'}
Calculated Class Weights (0=No Sepsis, 1=Sepsis): [0.14533333480358124, 1.8546667098999023]

Starting training on device: cpu
Epoch [1/20] | Train Loss: 0.6822 | Val Loss: 0.6293, Acc: 0.8237, Prec: 0.1969, Rec: 0.4633, AUC: 0.7302
Epoch [2/20] | Train Loss: 0.6222 | Val Loss: 0.5806, Acc: 0.8530, Prec: 0.2287, Rec: 0.4312, AUC: 0.7682
Epoch [